# Baseline

Author: Mathias Marryat Husted

In [2]:
# Imports and constants
import pandas as pd
import gzip


BENCHMARK_PATH = "../../data/data set/benchmark.csv"
CONTROL_PATH = "../../data/data set/control.csv"
# https://www.genenames.org/download/
HGNC_PATH = "../../data/baseline/hgnc_complete_set.txt"
# https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Extras/
PUBCHEM_DIR = "../../data/baseline"

# Initialize all the data in pandas
# Read everything as as string and turn off NA coercion
benchmark = pd.read_csv(BENCHMARK_PATH, sep=",", dtype=str, keep_default_na=False)
control = pd.read_csv(CONTROL_PATH, sep=",", dtype=str, keep_default_na=False)
hgnc_data = pd.read_csv(HGNC_PATH, sep="\t", dtype=str, keep_default_na=False)


This script performs the baseline runs B1, B2, B3, B4 taking the benchmark.csv or similar data set as the input. The data sets mentioned above must be present in the directories


In [89]:
# Double check gene coverage of HGNC symbol == Control canonical_name
def coverage_check(hgnc_data, control):
    control_genes = control[control["entity_type"] == "gene"]
    merged_df = pd.merge(
        left=control_genes,
        right=hgnc_data,
        left_on="hgnc_id",
        right_on="hgnc_id",
        how="left",
        indicator=True,
    )

    not_covered = merged_df[merged_df["_merge"] == "left_only"]
    print(f"Gene coverage check {"passed" if len(not_covered) == 0 else "failed"}, {len(merged_df)} total items")
    assert len(not_covered) == 0, f"{len(not_covered)} IDs may be non-existent!"
    mismatched = merged_df[merged_df["symbol"] != merged_df["canonical_name"]]
    assert len(mismatched) == 0, f"The HGNC symbol does not match the canonical name on {len(mismatched)} rows!"
    assert len(merged_df) == len(control_genes), "Row count changed after merge"

    return merged_df

coverage_check(hgnc_data, control)

Gene coverage check passed, 80 total items


,id,input,canonical_name,entity_type,id_status,pubchem_id,hgnc_id,uniprot_id,ncbi_gene_id,accepted_id,...,lncrnadb,enzyme_id,intermediate_filament_db,rna_central_id,lncipedia,gtrnadb,agr,mane_select,gencc,_merge
0,CTRL-009,ABCB1,ABCB1,gene,single,,HGNC:40,P08183,5243,HGNC:40,...,,7.6.2.1,,,,,HGNC:40,ENST00000622132.5|NM_001348946.2,HGNC:40,both
1,CTRL-010,ACE,ACE,gene,single,,HGNC:2707,P12821,1636,HGNC:2707,...,,3.4.15.1,,,,,HGNC:2707,ENST00000290866.10|NM_000789.4,HGNC:2707,both
2,CTRL-011,ACTB,ACTB,gene,single,,HGNC:132,P60709,60,HGNC:132,...,,,,,,,HGNC:132,ENST00000646664.1|NM_001101.5,HGNC:132,both
3,CTRL-012,APOE,APOE,gene,single,,HGNC:613,P02649,348,HGNC:613,...,,,,,,,HGNC:613,ENST00000252486.9|NM_000041.4,HGNC:613,both
4,CTRL-013,APP,APP,gene,single,,HGNC:620,P05067,351,HGNC:620,...,,,,,,,HGNC:620,ENST00000346798.8|NM_000484.4,HGNC:620,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,CTRL-107,TGFB1,TGFB1,gene,single,,HGNC:11766,P01137,7040,HGNC:11766,...,,,,,,,HGNC:11766,ENST00000221930.6|NM_000660.7,HGNC:11766,both
76,CTRL-108,TNF,TNF,gene,single,,HGNC:11892,P01375,7124,HGNC:11892,...,,,,,,,HGNC:11892,ENST00000449264.3|NM_000594.4,,both
77,CTRL-109,TP53,TP53,gene,single,,HGNC:11998,P04637,7157,HGNC:11998,...,,,,,,,HGNC:11998,ENST00000269305.9|NM_000546.6,HGNC:11998,both
78,CTRL-110,TUBB,TUBB,gene,single,,HGNC:20778,P07437,203068,HGNC:20778,...,,,,,,,HGNC:20778,ENST00000327892.13|NM_178014.4,HGNC:20778,both


# B1: Exact lookup against DB lookup
Requires: Database download of both HGNC as well as PubChem

In [4]:
# Run the benchmark on genes
def run_b1a(hgnc_data, benchmark):
    # Read only approved symbols
    hgnc_data = hgnc_data[hgnc_data["status"] == "Approved"]
    hgnc_data = hgnc_data[hgnc_data["symbol"] != ""]
    benchmark = benchmark[benchmark["entity_type"] == "gene"] # Gold entity type used for the benchmark
    # Approved symbols are unique, so this is a plain 1:1 lookup
    assert hgnc_data["symbol"].is_unique, "duplicate approved symbols in the HGNC data"
    symbol_index = hgnc_data.set_index("symbol")["hgnc_id"].to_dict()
    print(f"{HGNC_PATH}: {len(hgnc_data)} approved records, {len(symbol_index)} symbols")
    
    # Look up gene data
    results = []
    for row in benchmark.itertuples():
        # The actual B1 baseline check: exact lookup of the raw, noisy input
        hgnc_id = symbol_index.get(row.input)

        results.append({
            "id": row.id,
            "canonical_name": row.canonical_name,
            "input": row.input,
            "n_candidates": int(hgnc_id is not None),
            "noise_type": row.noise_type,
            "id_status": row.id_status,
            "accepted_id": row.accepted_id,
            "returned_id": hgnc_id,
        })

    return pd.DataFrame(results)


In [5]:


b1_results = run_b1a(hgnc_data, benchmark)

matched_raw_input = b1_results["returned_id"].notna().sum()

print(f"{matched_raw_input} candidates were matched on their raw input")

coverage_check(hgnc_data, control)

b1_results


../../data/baseline/hgnc_complete_set.txt: 45083 approved records, 45083 symbols
0 candidates were matched on their raw input
Gene coverage check passed, 80 total items


,id,canonical_name,input,n_candidates,noise_type,id_status,accepted_id,returned_id
0,GENE-001,TP53,TP35,0,typo,single,HGNC:11998,None
1,GENE-002,EGFR,EFGR,0,typo,single,HGNC:3236,None
2,GENE-003,BRCA1,BRAC1,0,typo,single,HGNC:1100,None
3,GENE-004,CFTR,CFRT,0,typo,single,HGNC:1884,None
4,GENE-005,KRAS,KARS,0,typo,ambiguous,HGNC:6407,None
...,...,...,...,...,...,...,...,...
152,GENE-153,,CDK94,0,non_existent,none,,None
153,GENE-154,,MAPK47,0,non_existent,none,,None
154,GENE-155,,IL73,0,non_existent,none,,None
155,GENE-156,,SLC6A99,0,non_existent,none,,None


Change: Add case folding because the synonym file stores some chemicals in all-caps, others capitalized etc.

- Chloroform -> CHLOROFORM
- Melatonin -> Melatonin
- Hexane -> HEXANE

In [ ]:
def read_pubchem(path, search_names, gold_cids):
    name_matches = []
    gold_rows = []
    invalid_lines = 0
    folded_names = {n.casefold() for n in search_names}
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            elements = line.rstrip("\n").split("\t", 1)
            if len(elements) != 2:
                invalid_lines += 1
                continue
            cid, name = elements
            if name.casefold() in folded_names:
                name_matches.append((cid, name))
            if cid in gold_cids:
                gold_rows.append((cid, name))
    print(f"Number of invalid lines skipped: {invalid_lines}")
    return (pd.DataFrame(name_matches, columns=["cid", "name"]),
            pd.DataFrame(gold_rows, columns=["cid", "name"]))

benchmark_chemicals = benchmark[benchmark["entity_type"] == "chemical"]
control_chemicals = control[control["entity_type"] == "chemical"]

# Search names are all unique inputs across the union of the benchmark and control input columns (for control, it's equal to canonical_name)
search_names = set(benchmark_chemicals["input"]) | set(control_chemicals["canonical_name"])


print(f"# chemicals in benchmark: {len(benchmark_chemicals)}, # chemicals with a unique name in the union: {len(search_names)}")


gold_cids = set(control_chemicals["pubchem_id"]) - {""}

print(f"# chemicals in control: {len(control_chemicals)}, # chemicals with a PubChem ID: {len(gold_cids)}")

# chemicals in benchmark: 162, # chemicals with a unique name in the union: 264
# chemicals in control: 109, # chemicals with a PubChem ID: 108


In [9]:
# Execute the read_pubchem function for the CID-Title.gz file. Match once on title (from the input chemicals) and once on
# the pre-verified PubChem IDs (expect all to match)
# Write the result to a CSV for later reference
title_hits, title_gold = read_pubchem(f"{PUBCHEM_DIR}/CID-Title.gz", search_names, gold_cids)

title_hits.to_csv("../results/title_hits.csv", index=False)
title_gold.to_csv("../results/title_gold.csv", index=False)

Number of invalid lines skipped: 0


In [10]:
# Now we'll do the same but for the filtered synonym file
synonym_hits, synonym_gold = read_pubchem(f"{PUBCHEM_DIR}/CID-Synonym-filtered.gz", search_names, gold_cids)

synonym_hits.to_csv("../results/synonym_hits.csv", index=False)
synonym_gold.to_csv("../results/synonym_gold.csv", index=False)

Number of invalid lines skipped: 0


- 892,An inositol (Matches up in PubChem, generic inositol in myo- configuration)
- 6134,beta-lactose: Needs further clarification

Analyze the data from our function. Make sure you've generated all the CSVs before continuing here

In [86]:
def run_b1b_chemicals(hits_file, chemicals):
    data = pd.read_csv(hits_file, sep=",", dtype=str, keep_default_na=False)
    data["folded"] = data["name"].str.casefold()

    results = []
    for row in chemicals.itertuples():
        matches = data[data["folded"] == row.input.casefold()]
        n_cand = matches["cid"].nunique() # Is the ID unique?
        returned_id = matches["cid"].iloc[0] if n_cand == 1 else None # Return the matched ID if it's unique
        results.append({
            "id": row.id,
            "canonical_name": row.canonical_name,
            "input": row.input,
            "n_candidates": n_cand,
            "noise_type": row.noise_type,
            "id_status": row.id_status,
            "accepted_id": row.accepted_id,
            "returned_id": returned_id,
        })
    return pd.DataFrame(results)

b1b_chemicals_result = run_b1b_chemicals("../results/synonym_hits.csv", control_chemicals.assign(noise_type="control"))

print(f"Number of chemicals with exactly 1 ID: {b1b_chemicals_result["returned_id"].notna().sum()}/{b1b_chemicals_result.shape[0]}")
assert (b1b_chemicals_result["returned_id"] == b1b_chemicals_result["accepted_id"]).sum() == 107, "Not all IDs matched!"
# Show the outliers
b1b_chemicals_result[b1b_chemicals_result["n_candidates"] != 1]


Number of chemicals with exactly 1 ID: 107/109


,id,canonical_name,input,n_candidates,noise_type,id_status,accepted_id,returned_id
13,CTRL-035,D-glucose 6-phosphate,D-glucose 6-phosphate,2,control,ambiguous,439958;99058,NaN
89,CTRL-170,phosphatidylcholine,phosphatidylcholine,0,control,none,,NaN
